# 🦵 KneeGuard AI — Knee Acoustic Classifier (WAV Edition)

> **Updated Notebook** — trains directly on `.wav` files to match the web app's inference pipeline exactly.

**Data layout expected inside the zip:**
```
your_zip.zip
├── 0/          ← Normal WAV files
│   ├── 1.wav
│   └── ...
└── 1/          ← Abnormal WAV files
    ├── 1.wav
    └── ...
```

---
### ⚠️ Key Design Principle
The feature extraction here is **100% identical** to `web_app/backend/feature_extractor.py`.
Signal tiling (not zero-padding) is used for short clips so spectral features remain meaningful.

## Step 1 — Install Dependencies

In [ ]:
!pip install librosa scikit-learn xgboost imbalanced-learn joblib matplotlib seaborn tqdm -q
print('✅ All packages installed!')

## Step 2 — Upload Your WAV Zip

In [ ]:
from google.colab import files
import zipfile, os

print('📁 Upload your zip file containing 0/ and 1/ folders of .wav files...')
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
print(f'\n📦 Extracting {zip_name}...')
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/dataset')
print('✅ Extracted to /content/dataset')

# Auto-find the folder containing 0/ and 1/
BASE = '/content/dataset'
for root, dirs, _ in os.walk(BASE):
    if '0' in dirs and '1' in dirs:
        BASE = root
        break

normal_dir   = os.path.join(BASE, '0')
abnormal_dir = os.path.join(BASE, '1')

normal_wavs   = [f for f in os.listdir(normal_dir)   if f.endswith('.wav')]
abnormal_wavs = [f for f in os.listdir(abnormal_dir) if f.endswith('.wav')]

print(f'\n📂 Data base path  : {BASE}')
print(f'   ✅ Normal   (0/) : {len(normal_wavs)} .wav files')
print(f'   ⚠️  Abnormal (1/) : {len(abnormal_wavs)} .wav files')

## Step 3 — Inspect a WAV File

In [ ]:
import librosa
import numpy as np
import matplotlib.pyplot as plt

SR = 22050  # Fixed sample rate for all processing

sample_path = os.path.join(normal_dir, normal_wavs[0])
signal, sr_orig = librosa.load(sample_path, sr=SR, mono=True)

print(f'📄 File          : {normal_wavs[0]}')
print(f'   Original SR   : {sr_orig} Hz')
print(f'   Samples       : {len(signal)}')
print(f'   Duration      : {len(signal)/SR*1000:.1f} ms')
print(f'   Amplitude range: [{signal.min():.4f}, {signal.max():.4f}]')

plt.figure(figsize=(12, 3))
plt.plot(signal, color='#00e5ff', linewidth=0.8)
plt.title(f'Sample Normal Signal — {normal_wavs[0]}', fontsize=12)
plt.xlabel('Sample index'); plt.ylabel('Amplitude')
plt.tight_layout(); plt.show()

## Step 4 — Feature Extraction Pipeline

> **⚠️ This must exactly match `web_app/backend/feature_extractor.py`**

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────
# CRITICAL: This function is IDENTICAL to feature_extractor.py
# in the web app backend. Do NOT change one without the other.
# ─────────────────────────────────────────────────────────────
MIN_SAMPLES = 4096  # Same as app.py

def preprocess_signal(signal: np.ndarray) -> np.ndarray:
    """Tile short signals to MIN_SAMPLES. Preserves spectral content."""
    signal = signal.astype(np.float32)
    if len(signal) < MIN_SAMPLES:
        repeats = int(np.ceil(MIN_SAMPLES / len(signal)))
        signal = np.tile(signal, repeats)[:MIN_SAMPLES]
    return signal

def extract_features(signal: np.ndarray, sr: int = SR) -> np.ndarray:
    """Extract 72 acoustic features from a 1-D signal."""
    signal = preprocess_signal(signal)

    # Z-score normalize
    if signal.max() != signal.min():
        signal = (signal - signal.mean()) / (signal.std() + 1e-9)

    features = []

    # 1. MFCC (13 × mean+std = 26)
    mfcc = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=13)
    features.extend(np.mean(mfcc, axis=1))
    features.extend(np.std(mfcc, axis=1))

    # 2. Spectral Centroid (2)
    sc = librosa.feature.spectral_centroid(y=signal, sr=sr)[0]
    features.extend([np.mean(sc), np.std(sc)])

    # 3. Spectral Bandwidth (2)
    sb = librosa.feature.spectral_bandwidth(y=signal, sr=sr)[0]
    features.extend([np.mean(sb), np.std(sb)])

    # 4. Spectral Rolloff (2)
    sr_feat = librosa.feature.spectral_rolloff(y=signal, sr=sr)[0]
    features.extend([np.mean(sr_feat), np.std(sr_feat)])

    # 5. Zero Crossing Rate (2)
    zcr = librosa.feature.zero_crossing_rate(signal)[0]
    features.extend([np.mean(zcr), np.std(zcr)])

    # 6. RMS Energy (2)
    rms = librosa.feature.rms(y=signal)[0]
    features.extend([np.mean(rms), np.std(rms)])

    # 7. Chroma STFT (12 × mean+std = 24)
    chroma = librosa.feature.chroma_stft(y=signal, sr=sr)
    features.extend(np.mean(chroma, axis=1))
    features.extend(np.std(chroma, axis=1))

    # 8. Spectral Contrast (7 × mean+std = 14)
    contrast = librosa.feature.spectral_contrast(y=signal, sr=sr)
    features.extend(np.mean(contrast, axis=1))
    features.extend(np.std(contrast, axis=1))

    return np.array(features, dtype=np.float32)

# Test on one file
test_feats = extract_features(signal)
print(f'✅ Feature vector size: {len(test_feats)}')
print(f'   (should be 72)')

## Step 5 — Load All WAV Files & Extract Features

In [ ]:
from tqdm import tqdm

def load_class_wav(folder: str, label: int):
    X, y = [], []
    wavs = [f for f in os.listdir(folder) if f.endswith('.wav')]
    for fname in tqdm(wavs, desc=f'Class {label} ({"Normal" if label==0 else "Abnormal"})'):
        try:
            sig, _ = librosa.load(os.path.join(folder, fname), sr=SR, mono=True)
            if len(sig) == 0:
                continue
            X.append(extract_features(sig))
            y.append(label)
        except Exception as e:
            print(f'  ⚠️ Skipped {fname}: {e}')
    return X, y

print('🔄 Loading Normal (class 0)...')
X0, y0 = load_class_wav(normal_dir, 0)
print(f'   → {len(X0)} samples\n')

print('🔄 Loading Abnormal (class 1)...')
X1, y1 = load_class_wav(abnormal_dir, 1)
print(f'   → {len(X1)} samples')

X = np.array(X0 + X1)
y = np.array(y0 + y1)

print(f'\n✅ Dataset: X={X.shape}, y={y.shape}')
print(f'   Normal={sum(y==0)}, Abnormal={sum(y==1)}')
print(f'   Imbalance ratio: {sum(y==0)/sum(y==1):.1f}x')

## Step 6 — EDA: Compare Normal vs Abnormal Features

In [ ]:
import seaborn as sns
import pandas as pd

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Class distribution
axes[0].bar(['Normal (0)', 'Abnormal (1)'], [sum(y==0), sum(y==1)],
            color=['#06d6a0', '#ef476f'])
axes[0].set_title('Class Distribution', fontsize=12)
for i, v in enumerate([sum(y==0), sum(y==1)]):
    axes[0].text(i, v+2, str(v), ha='center', fontweight='bold')

# MFCC 1 distribution by class
df = pd.DataFrame({'MFCC_1': X[:,0], 'label': ['Normal' if l==0 else 'Abnormal' for l in y]})
df[df.label=='Normal']['MFCC_1'].hist(bins=40, alpha=0.7, ax=axes[1], color='#06d6a0', label='Normal')
df[df.label=='Abnormal']['MFCC_1'].hist(bins=40, alpha=0.7, ax=axes[1], color='#ef476f', label='Abnormal')
axes[1].legend()
axes[1].set_title('MFCC-1 Distribution (Normal vs Abnormal)', fontsize=12)
axes[1].set_xlabel('MFCC-1 Mean')

plt.tight_layout(); plt.show()
print('✅ EDA complete')

## Step 7 — Train/Test Split + SMOTE Oversampling

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from imblearn.over_sampling import SMOTE
import joblib

# Stratified split (preserves class ratio in test set)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# SMOTE — oversample Abnormal class in training set only
smote = SMOTE(random_state=42, k_neighbors=min(5, sum(y_train==1)-1))
X_train_bal, y_train_bal = smote.fit_resample(X_train_sc, y_train)

print(f'Train (before SMOTE): Normal={sum(y_train==0)}, Abnormal={sum(y_train==1)}')
print(f'Train (after SMOTE) : Normal={sum(y_train_bal==0)}, Abnormal={sum(y_train_bal==1)}')
print(f'Test  (untouched)   : Normal={sum(y_test==0)}, Abnormal={sum(y_test==1)}')

## Step 8 — Train & Compare Models

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
import time

models = {
    'Random Forest': RandomForestClassifier(
        n_estimators=300, max_depth=15, min_samples_leaf=2,
        class_weight='balanced', random_state=42, n_jobs=-1
    ),
    'SVM (RBF)': SVC(
        kernel='rbf', C=10, gamma='scale',
        probability=True, random_state=42
    ),
    'XGBoost': XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='logloss', random_state=42, n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42
    ),
}

results = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    print(f'\n⏳ Training {name}...')
    t0 = time.time()

    # CV on SMOTE-balanced training set
    scores = cross_val_score(model, X_train_bal, y_train_bal, cv=cv,
                             scoring='f1_weighted', n_jobs=-1)
    model.fit(X_train_bal, y_train_bal)
    elapsed = time.time() - t0

    y_pred = model.predict(X_test_sc)   # test on ORIGINAL (unbalanced) test set
    y_prob = model.predict_proba(X_test_sc)[:, 1]
    report = classification_report(y_test, y_pred,
                                   target_names=['Normal','Abnormal'], output_dict=True)
    auc = roc_auc_score(y_test, y_prob)

    results[name] = dict(model=model, cv_f1=scores.mean(), cv_std=scores.std(),
                         test_f1=report['weighted avg']['f1-score'],
                         test_acc=report['accuracy'],
                         abnormal_recall=report['Abnormal']['recall'],
                         auc=auc, time=elapsed)

    print(f'   CV F1        : {scores.mean():.4f} ± {scores.std():.4f}')
    print(f'   Test Accuracy: {report["accuracy"]*100:.2f}%')
    print(f'   Abnormal Recall: {report["Abnormal"]["recall"]*100:.1f}%')
    print(f'   ROC-AUC      : {auc:.4f}  |  Time: {elapsed:.1f}s')

## Step 9 — Select Best Model & Full Report

In [ ]:
# ── Select best model by ROC-AUC (primary) + Abnormal recall (secondary)
# We prefer a model that catches Abnormals, not just maximizes accuracy
best_name = max(results, key=lambda n: (results[n]['auc'], results[n]['abnormal_recall']))
best_model = results[best_name]['model']

print('=' * 55)
print(f'  🏆 BEST MODEL  : {best_name}')
print(f'  ROC-AUC        : {results[best_name]["auc"]:.4f}')
print(f'  Test Accuracy  : {results[best_name]["test_acc"]*100:.2f}%')
print(f'  Abnormal Recall: {results[best_name]["abnormal_recall"]*100:.1f}%')
print('=' * 55)

y_pred_best = best_model.predict(X_test_sc)
print('\n📊 Full Classification Report:')
print(classification_report(y_test, y_pred_best, target_names=['Normal','Abnormal']))

cm = confusion_matrix(y_test, y_pred_best)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax[0],
            xticklabels=['Normal','Abnormal'], yticklabels=['Normal','Abnormal'])
ax[0].set_title(f'Confusion Matrix — {best_name}', fontsize=11)
ax[0].set_ylabel('True'); ax[0].set_xlabel('Predicted')

names_list = list(results.keys())
aucs = [results[n]['auc'] for n in names_list]
colors = ['#ef476f' if n == best_name else '#4361ee' for n in names_list]
bars = ax[1].bar(names_list, aucs, color=colors, edgecolor='white')
ax[1].set_title('Model Comparison (ROC-AUC)', fontsize=11)
ax[1].set_ylim(0.4, 1.0)
ax[1].set_xticklabels(names_list, rotation=15, ha='right')
for bar, v in zip(bars, aucs):
    ax[1].text(bar.get_x()+bar.get_width()/2, v+0.01, f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')
plt.tight_layout(); plt.savefig('/content/results.png', dpi=120); plt.show()

## Step 10 — Save Model Files

In [ ]:
import json

joblib.dump(best_model, '/content/knee_model.pkl')
joblib.dump(scaler,     '/content/knee_scaler.pkl')

meta = {
    'model_name'    : best_name,
    'feature_count' : int(X.shape[1]),
    'sample_rate'   : SR,
    'min_samples'   : MIN_SAMPLES,
    'classes'       : {0: 'Normal', 1: 'Abnormal'},
    'test_accuracy' : float(results[best_name]['test_acc']),
    'roc_auc'       : float(results[best_name]['auc']),
    'abnormal_recall': float(results[best_name]['abnormal_recall']),
    'trained_on'    : 'wav_files',
}
with open('/content/knee_model_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('✅ Saved: knee_model.pkl, knee_scaler.pkl, knee_model_meta.json')
print(f'\n📋 Metadata:\n{json.dumps(meta, indent=2)}')

## Step 11 — Download Model Files ⬇️

In [ ]:
from google.colab import files

print('⬇️  Downloading 3 files...')
files.download('/content/knee_model.pkl')
files.download('/content/knee_scaler.pkl')
files.download('/content/knee_model_meta.json')

print('\n✅ Download complete!')
print('📌 Place all 3 files in → web_app/backend/')
print('   Then restart uvicorn and test again.')

## Step 12 — Quick Test on Uploaded WAV

In [ ]:
print('📁 Upload a test .wav file to verify...')
uploaded_test = files.upload()

for fname in uploaded_test:
    sig, _ = librosa.load(fname, sr=SR, mono=True)
    feats = extract_features(sig)
    feats_sc = scaler.transform(feats.reshape(1, -1))
    pred = int(best_model.predict(feats_sc)[0])
    prob = best_model.predict_proba(feats_sc)[0]
    label = 'Normal' if pred == 0 else 'Abnormal'
    icon = '✅' if pred == 0 else '⚠️'
    print(f'\n{icon}  File        : {fname}')
    print(f'   Prediction  : {label}')
    print(f'   Confidence  : {prob[pred]*100:.1f}%')
    print(f'   P(Normal)   : {prob[0]*100:.1f}%')
    print(f'   P(Abnormal) : {prob[1]*100:.1f}%')